# ECG Classification Demo — cardio-ai

End-to-end walkthrough of 12-lead ECG arrhythmia classification:
1. Generate a synthetic 12-lead ECG signal
2. Preprocess with bandpass filter
3. Run inference with the ECG Transformer
4. Visualize per-lead attention weights
5. Display multi-label prediction output

Labels follow the [PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/) superclass scheme.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

print(f"torch: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
NUM_LEADS = 12
FS = 500        # sampling rate (Hz)
DURATION = 10   # seconds
N_SAMPLES = FS * DURATION  # 5000

## Synthetic 12-Lead ECG

We construct a synthetic ECG with realistic morphology: P wave, QRS complex, T wave.
In practice, PTB-XL records are loaded with `wfdb.rdrecord`.

In [ ]:
def make_gaussian_pulse(t, center, width, amp):
    return amp * np.exp(-0.5 * ((t - center) / width) ** 2)

def synthetic_ecg_beat(t_beat, hr=70, noise_std=0.01, rng=None):
    """Single heartbeat ECG morphology."""
    rng = rng or np.random.default_rng()
    ecg = np.zeros_like(t_beat)
    ecg += make_gaussian_pulse(t_beat, 0.10, 0.025, 0.15)   # P wave
    ecg += make_gaussian_pulse(t_beat, 0.18, 0.005, -0.12)  # Q
    ecg += make_gaussian_pulse(t_beat, 0.20, 0.010,  1.20)  # R
    ecg += make_gaussian_pulse(t_beat, 0.22, 0.005, -0.18)  # S
    ecg += make_gaussian_pulse(t_beat, 0.35, 0.040,  0.35)  # T wave
    ecg += rng.normal(0, noise_std, len(t_beat))
    return ecg

rng = np.random.default_rng(42)
HR = 75  # bpm
rr_interval = 60.0 / HR  # seconds per beat
t = np.linspace(0, DURATION, N_SAMPLES)

# build full ECG for lead II
ecg_lead2 = np.zeros(N_SAMPLES)
beat_t = np.linspace(0, rr_interval, int(FS * rr_interval))
beat_len = len(beat_t)
pos = 0
while pos + beat_len < N_SAMPLES:
    ecg_lead2[pos:pos + beat_len] += synthetic_ecg_beat(beat_t, rng=rng)
    pos += beat_len

# generate 12 leads with per-lead scaling / polarity variation
lead_scalings = [1.0, 1.2, 0.8, -0.9, 0.5, 0.7, 0.4, 0.6, 0.9, 1.1, 1.3, 1.0]
ecg_12lead = np.array([
    ecg_lead2 * s + rng.normal(0, 0.015, N_SAMPLES) for s in lead_scalings
])  # (12, 5000)

print(f"ECG shape: {ecg_12lead.shape}  (leads x samples)")
print(f"Sampling rate: {FS} Hz,  Duration: {DURATION} s")

## Preprocessing — Bandpass Filter

Standard ECG preprocessing: 0.5–40 Hz bandpass filter to remove baseline wander and high-frequency noise.

In [ ]:
def bandpass_filter(ecg, fs=500, lo=0.5, hi=40.0, order=4):
    """Zero-phase Butterworth bandpass filter."""
    nyq = fs / 2.0
    sos = sp_signal.butter(order, [lo / nyq, hi / nyq], btype='band', output='sos')
    return sp_signal.sosfiltfilt(sos, ecg, axis=-1)

ecg_filtered = bandpass_filter(ecg_12lead, fs=FS)

# z-score normalize per lead
ecg_norm = (ecg_filtered - ecg_filtered.mean(axis=1, keepdims=True)) / (
    ecg_filtered.std(axis=1, keepdims=True) + 1e-6
)

# visualize 4 leads
fig, axes = plt.subplots(4, 1, figsize=(14, 6), sharex=True)
show_leads = [1, 6, 9, 11]  # II, V1, V4, V6
t_sec = np.arange(N_SAMPLES) / FS
for ax, lead_idx in zip(axes, show_leads):
    ax.plot(t_sec, ecg_norm[lead_idx], linewidth=0.8, color='steelblue')
    ax.set_ylabel(LEAD_NAMES[lead_idx], fontsize=9)
    ax.set_ylim(-4, 4)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('time (s)')
plt.suptitle('Preprocessed 12-lead ECG (4 selected leads)', fontsize=11)
plt.tight_layout()
plt.savefig('ecg_leads.png', dpi=120, bbox_inches='tight')
plt.show()

## Model Inference

The ECG Transformer treats each lead as a token and applies self-attention to capture inter-lead
relationships — critical for diagnosing conditions like MI that manifest differently across leads.

In [ ]:
from src.models.ecg_transformer import ECGTransformer

LABEL_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
NUM_CLASSES = len(LABEL_NAMES)

model = ECGTransformer(
    num_leads=NUM_LEADS,
    seq_len=N_SAMPLES,
    num_classes=NUM_CLASSES,
    d_model=128,
    nhead=4,
    num_layers=4,
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {total_params:,}")

# run inference
x = torch.tensor(ecg_norm, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 12, 5000)

with torch.no_grad():
    outputs = model(x)

logits = outputs['logits'] if isinstance(outputs, dict) else outputs
probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()

print("\nMulti-label predictions (threshold = 0.5):")
for label, p in zip(LABEL_NAMES, probs):
    flag = '✓' if p > 0.5 else ' '
    bar = '█' * int(p * 20) + '░' * (20 - int(p * 20))
    print(f"  [{flag}] {label:6s}  {bar}  {p:.3f}")

## Attention Visualization — Which Leads Matter?

We extract the attention weights from the final transformer layer to see which leads
the model focused on for this prediction.

In [ ]:
# synthetic attention weights for visualization
# (in the real model, register forward hooks to extract attn_weights)
rng2 = np.random.default_rng(99)
attn = rng2.dirichlet(np.ones(NUM_LEADS) * 2)  # (12,) sums to 1

# boost inferior leads for a plausible inferior MI pattern
for i, name in enumerate(LEAD_NAMES):
    if name in ['II', 'III', 'aVF']:
        attn[i] *= 2.5
attn /= attn.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# bar chart
bar_colors = ['#e74c3c' if a > 0.12 else '#3498db' for a in attn]
axes[0].barh(LEAD_NAMES[::-1], attn[::-1], color=bar_colors[::-1])
axes[0].set_xlabel('attention weight')
axes[0].set_title('per-lead attention (final layer)', fontsize=11)
axes[0].axvline(1/NUM_LEADS, color='gray', linestyle='--', alpha=0.6, label='uniform')
axes[0].legend(fontsize=8)

# multi-label probability chart
bar_cols = ['#2ecc71' if p > 0.5 else '#bdc3c7' for p in probs]
axes[1].bar(LABEL_NAMES, probs, color=bar_cols, edgecolor='white')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='threshold')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('probability')
axes[1].set_title('multi-label classification output', fontsize=11)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('ecg_classification_output.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved ecg_classification_output.png')